# 02 Trimmed Data Ingestion & Preprocessing Workbook

### Overview
This notebook contains the trimmed data preperation and preprocessing pipeline. This contains only the essential cells for importing the raw mp3 files from the Free Music Archive (FMA) small subset and transforming them into mel spectrograms. This creates the relevant processed mel spectrogam files and split CSV files for EDA and modeling work.

For the full data ingestion and preprocessing work performed, including exploratory visualizations, comparisons, and plotting please refer to `notebooks/02_DataPreProcessing_Trimmed.ipynb`.

#### Prerequisites:
* Run the appropriate repo set-up instructions detailed in the project README.md.
* Run `01_SetupDataFolder.ipynb` first to extract `fma_small.zip` into `data/local/`.

#### Preprocessing Methods:
* Ingested raw mp3 and tracks metadata from FMA small subset. Data already subsetted by FMA into training, validation, and testing splits.
* Leveraging librosa library, loaded mp3 files at a audio sample rate of 22,050 Hz and standardized files to exactly 30 seconds.
* Via librosa, then converted to mel spectrograms, using 128 mel frequency bands, fmax of 8,000 Hz, n_fft of 2,048, and hop_length of 512. This yields uniform results with shape `(128, 1292)`.
* Power compressed decibels relative to peak of 80, resulting in a range of (-80.0 db, 0.0 db).

#### Outputs:
* `data/local/processed_mel/*.npy` (skips if files already exist)
* `data/tracked/train_mel.csv`, `val_mel.csv`, `test_mel.csv`
* `data/tracked/genre_mapping.csv`

#### Results:
* 8000 mp3 tracks ingested from FMA -> split into: 
    * Training: 6393
    * Validation: 800
    * Test: 800
* 6 tracks failed mel spectrogram conversion, all in the training dataset. Given, the small number and all being from the largest dataset, removal acceptable and will not use in EDA or modeling.
* 1 track removed purposefully for containing 0 audio (completely silent), as it was determined to be an outlier and likely unuseful for modeling.
* No artist overlap detected between training, validation, and test datasets.
* However, album overlap detected across datasets, with 90 albums overlapping across splits. Flagging for deeper analysis via EDA.

#### Step 1: Import Libraries

In [1]:
from pathlib import Path
from dataclasses import dataclass
from time import perf_counter
import numpy as np
import pandas as pd
import librosa
import os

#### Step 2: Define Configs and Data Ingestion
***Note: Follow Directions in notebooks/01_SetupDataFolder.ipynb first as downloading and setting up mp3 data is required.***

In [2]:
# Define Config
@dataclass(frozen=True)
class Config:
    audio_dir: Path = Path("../data/local/fma_small_mp3/fma_small")
    tracks_csv: Path = Path("../data/local/fma_metadata/tracks.csv")
    output_dir: Path = Path("../data/local/processed_mel") # Keeping processed mel spectos local given the size of data
    n_mels: int = 128
    fmax: int = 8000
    n_fft: int = 2048
    hop_length: int = 512
    sr: int = 22050
    top_db: int = 80
    max_tracks: int | None = None  # Set a max number of tracks to process - benchmark first then expand to full after assessment of time and resources. Set to None to process all tracks.

cfg = Config()

In [3]:
# Data Ingestion
tracks = pd.read_csv(cfg.tracks_csv, header=[0, 1], index_col=0, low_memory=False)
small = tracks.loc[tracks[("set", "subset")] == "small",
                   [("track", "genre_top"), ("set", "split")]].copy()
small.columns = ["genre", "split"]
small.index = small.index.astype(int)
small.index.name = "track_id"
small["audio_path"] = [cfg.audio_dir / f"{tid:06d}"[:3] / f"{tid:06d}.mp3"
                       for tid in small.index]

print("Metadata tracks:", len(small))
print("Split counts:\n", small.groupby(["split", "genre"]).size().unstack(fill_value=0))
print("Missing labels:", small.genre.isna().sum())
print("Missing split assignments:", small.split.isna().sum())
print("Missing files:", (~small.audio_path.map(Path.is_file)).sum())

Metadata tracks: 8000
Split counts:
 genre       Electronic  Experimental  Folk  Hip-Hop  Instrumental  \
split                                                               
test               100           100   100      100           100   
training           800           800   800      800           800   
validation         100           100   100      100           100   

genre       International  Pop  Rock  
split                                 
test                  100  100   100  
training              800  800   800  
validation            100  100   100  
Missing labels: 0
Missing split assignments: 0
Missing files: 0


#### Step 3: mp3_to_melspec(path) Function to Convert WAV to Mel Spectogram

In [4]:
def mp3_to_melspec(path):
    # librosa loads MP3 through the installed audio backend
    try:
        y, sr = librosa.load(path, sr=cfg.sr, mono=True)

        if y.size == 0 or not np.isfinite(y).all():
            raise ValueError("Audio is empty or contains non-finite samples.")

        # Also check for complete silence (all zeros) in the audio signal - will remove 1 track (track 107535) from the dataset
        if np.all(y == 0):
            raise ValueError("Audio is completely silent (all zeros).")

        # Measure before padding/truncation
        loaded_duration = len(y) / sr

        # Make every waveform exactly 30 seconds
        y = librosa.util.fix_length(y, size=sr * 30)
    
        # Compute mel spectrogram
        # n_mels=128: number of mel frequency bands (more bands = more frequency detail)
        # fmax=8000: only consider frequencies up to 8000 Hz (covers most speech and music)
        S = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            fmax=cfg.fmax,
            center=True
        )

        # Convert power to decibels (dB) for visualisation
        # power_to_db (vs amplitude_to_db above) because melspectrogram returns power, not amplitude
        # ref=np.max, normalise relative to the loudest frequency (0 dB = loudest)
        S_db = librosa.power_to_db(S, ref=np.max, top_db=cfg.top_db)

        return y, sr, S, S_db, loaded_duration, None

    except Exception as exc:
        return None, None, None, None, np.nan, str(exc)

#### Step 4: Convert MP3s to Mel Spectogram

In [5]:
selected_ids = (
    small.index[:cfg.max_tracks]
    if cfg.max_tracks is not None
    else small.index
)

results = small.loc[selected_ids, ["genre", "split", "audio_path"]].copy()
results["loaded_duration_seconds"] = np.nan
results["mel_path"] = None
results["conversion_error"] = None
results["status"] = "pending"

cfg.output_dir.mkdir(parents=True, exist_ok=True)
manifest_path = cfg.output_dir / "conversion_results.csv"

# Check if previous manifest exists and load it to resume processing
if manifest_path.is_file():
    previous_results = pd.read_csv(manifest_path, index_col=0)
    for col in ["loaded_duration_seconds", "mel_path", "conversion_error", "status"]:
        if col in previous_results.columns:
            results[col] = previous_results[col].reindex(results.index)
    print("Existing manifest found with", len(previous_results), "records.")
    
expected_shape = (
    cfg.n_mels,
    1 + (cfg.sr * 30) // cfg.hop_length
)

total = len(results)
start = perf_counter()

print("Processing, total=", total, " tracks")
print("Output directory:", cfg.output_dir.resolve())

for count, track_id in enumerate(results.index, start=1):
    mel_path = cfg.output_dir / f"{int(track_id):06d}.npy"

    # Check if file already exists and marked successful
    if mel_path.is_file():
        results.at[track_id, "mel_path"] = str(mel_path)
        results.at[track_id, "status"] = "success"
        if pd.isna(results.at[track_id, "loaded_duration_seconds"]):
            results.at[track_id, "loaded_duration_seconds"] = 30.0
        continue
    
    audio_path = results.at[track_id, "audio_path"]

    try:
        y, sr, S, S_db, duration, error = mp3_to_melspec(audio_path)

        if error is not None:
            raise RuntimeError(error)

        results.at[track_id, "loaded_duration_seconds"] = duration
        mel = S_db.astype(np.float32)
        
        if mel.shape != expected_shape:
            raise ValueError(f"Unexpected shape: {mel.shape}")

        if not np.isfinite(mel).all():
            raise ValueError("Spectrogram contains NaN or infinity.")

        mel_path = cfg.output_dir / f"{int(track_id):06d}.npy"
        np.save(mel_path, mel)

        # Verify the file can be read
        saved = np.load(mel_path)

        results.at[track_id, "mel_path"] = str(mel_path)
        results.at[track_id, "status"] = "success"

    except Exception as exc:
        results.at[track_id, "conversion_error"] = str(exc)
        results.at[track_id, "status"] = "failed"

    if count % 100 == 0 or count == total:
        results.to_csv(manifest_path)

        elapsed = perf_counter() - start
        remaining_minutes = (elapsed / count) * (total - count) / 60

        succeeded = (results["status"] == "success").sum()
        failed = (results["status"] == "failed").sum()

        print(
            f"{count}/{total} | "
            f"success={succeeded}, failed={failed} | "
            f"elapsed={elapsed / 60:.1f} min | "
            f"estimated remaining={remaining_minutes:.1f} min"
        )

successful = results["status"] == "success"
failed_results = results.loc[results["status"] == "failed"]

print("Successfully converted:", successful.sum())
print("Failed conversions:", len(failed_results))
print("Expected mel shape:", expected_shape)
print("Max original loaded duration:", results.loc[successful, "loaded_duration_seconds"].max(), "seconds")
print("Min original loaded duration:", results.loc[successful, "loaded_duration_seconds"].min(), "seconds")
print("Saved processing record:", manifest_path)

display(failed_results[["audio_path", "conversion_error"]])

Existing manifest found with 8000 records.
Processing, total= 8000  tracks
Output directory: /Users/mattthomas/GitHub/datasci-207-final-project/data/local/processed_mel
Successfully converted: 7993
Failed conversions: 7
Expected mel shape: (128, 1292)
Max original loaded duration: 30.002721088435376 seconds
Min original loaded duration: 29.964580498866212 seconds
Saved processing record: ../data/local/processed_mel/conversion_results.csv


Note: Illegal Audio-MPEG-Header 0x00000000 at offset 33361.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 22401.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 63168.
Note: Trying to resync...
Note: Skipped 1024 bytes in input.
[src/libmpg123/parse.c:wetwork():1349] error: Giving up resync after 1024 bytes - your stream is not nice... (maybe increasing resync limit could help).
[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? D

,audio_path,conversion_error
track_id,,
98565,../data/local/fma_small_mp3/fma_small/098/0985...,Unspecified internal error.
98567,../data/local/fma_small_mp3/fma_small/098/0985...,Unspecified internal error.
98569,../data/local/fma_small_mp3/fma_small/098/0985...,Unspecified internal error.
99134,../data/local/fma_small_mp3/fma_small/099/0991...,Error opening '../data/local/fma_small_mp3/fma...
107535,../data/local/fma_small_mp3/fma_small/107/1075...,Audio is completely silent (all zeros).
108925,../data/local/fma_small_mp3/fma_small/108/1089...,Error opening '../data/local/fma_small_mp3/fma...
133297,../data/local/fma_small_mp3/fma_small/133/1332...,Error opening '../data/local/fma_small_mp3/fma...


### Step 5. Data Checks & Splitting

In [6]:
small_meta = tracks.loc[tracks[("set", "subset")] == "small"].copy()

# Extract artist and album IDs
artist_ids = small_meta[("artist", "id")]
album_ids = small_meta[("album", "id")]
splits = small_meta[("set", "split")]

# Create dfs for artists within each dataset (train, val, test)
train_artists = set(artist_ids[splits == "training"])
val_artists   = set(artist_ids[splits == "validation"])
test_artists  = set(artist_ids[splits == "test"])

# Check for intersections across datasets
train_val_leak = train_artists.intersection(val_artists)
train_test_leak = train_artists.intersection(test_artists)
val_test_leak = val_artists.intersection(test_artists)

# Print Results
print("Unique artists - Train:", len(train_artists), "Val:", len(val_artists), "Test:", len(test_artists))
print("Artist overlap Train vs Val:  ", len(train_val_leak))
print("Artist overlap Train vs Test: ", len(train_test_leak))
print("Artist overlap Val vs Test:   ", len(val_test_leak))

# Print final conclusion based on leak results
assert len(train_val_leak) == 0 and len(train_test_leak) == 0 and len(val_test_leak) == 0, \
    "INVESTIGATION NEEDED: Artists overlap across splits."
print("PASS: No artist leakage across splits. The FMA split is artist-conditional.")

# Create dfs for albums within each dataset (train, val, test)
train_albums = set(album_ids[splits == "training"])
val_albums   = set(album_ids[splits == "validation"])
test_albums  = set(album_ids[splits == "test"])

# Check for album intersections across datasets
train_val_album_leak = train_albums.intersection(val_albums)
train_test_album_leak = train_albums.intersection(test_albums)
val_test_album_leak = val_albums.intersection(test_albums)
overlapping_albums = (train_albums & val_albums) | (train_albums & test_albums) | (val_albums & test_albums)

# Print Results
print("Unique albums - Train:", len(train_albums), "Val:", len(val_albums), "Test:", len(test_albums))
print("Album overlap Train vs Val:  ", len(train_val_album_leak))
print("Album overlap Train vs Test: ", len(train_test_album_leak))
print("Album overlap Val vs Test:   ", len(val_test_album_leak))
print("Overlapping albums across splits:", len(overlapping_albums))

# Print final conclusion based on leak results
print("INVESTIGATION NEEDED: 90 Albums overlap across splits. Flagging for deeper analysis via EDA.")

Unique artists - Train: 1888 Val: 219 Test: 202
Artist overlap Train vs Val:   0
Artist overlap Train vs Test:  0
Artist overlap Val vs Test:    0
PASS: No artist leakage across splits. The FMA split is artist-conditional.
Unique albums - Train: 2010 Val: 278 Test: 285
Album overlap Train vs Val:   58
Album overlap Train vs Test:  49
Album overlap Val vs Test:    21
Overlapping albums across splits: 90
INVESTIGATION NEEDED: 90 Albums overlap across splits. Flagging for deeper analysis via EDA.


#### Step 6. Save converted Mel files and split into pre-IDed train, validation, test for EDA and Modeling

In [7]:
if cfg.max_tracks is not None:
    raise RuntimeError(
        "Not creating CSV exports if cfg.max_tracks is set so only a sample runs.", 
        "Set cfg.max_tracks = None to process the full dataset before exporting."
    )

ready_all = results.loc[results["status"] == "success"].copy()

# Create genre integer mapping for modeling
genre_names = sorted(ready_all["genre"].unique())
genre_to_id = {genre: idx for idx, genre in enumerate(genre_names)}
ready_all["genre_id"] = ready_all["genre"].map(genre_to_id)

# Add in artist and album metadata from tracks.csv
ready_all["artist_id"] = small_meta.loc[ready_all.index, ("artist", "id")]
ready_all["artist_name"] = small_meta.loc[ready_all.index, ("artist", "name")]
ready_all["album_id"] = small_meta.loc[ready_all.index, ("album", "id")]
ready_all["album_title"] = small_meta.loc[ready_all.index, ("album", "title")]

# Identify columns to export and tracked path location
cols_to_export = [
    "genre", 
    "genre_id", 
    "mel_path", 
    "artist_id", 
    "artist_name", 
    "album_id", 
    "album_title", 
    "loaded_duration_seconds"
]
tracked_dir = Path("../data/tracked")
tracked_dir.mkdir(parents=True, exist_ok=True)

train_rows = ready_all.loc[ready_all["split"] == "training"].copy()
val_rows = ready_all.loc[ready_all["split"] == "validation"].copy()
test_rows = ready_all.loc[ready_all["split"] == "test"].copy()

print("Training:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))

# Save the processed mel spectrograms and metadata to CSV files for each split
for split_name, filename in [
    ("training", "train_mel.csv"),
    ("validation", "val_mel.csv"),
    ("test", "test_mel.csv")
]:
    subset = ready_all.loc[ready_all["split"] == split_name, cols_to_export]
    out_path = tracked_dir / filename
    subset.to_csv(out_path, index_label="track_id")
    print(f"Saved, {filename}: {len(subset)} records -> {out_path}")

# Check how failed conversion affected genre counts
display(pd.crosstab(ready_all["split"], ready_all["genre"]))

# Save Genre Mapping for future reference
genre_mapping_df = pd.DataFrame(
    list(genre_to_id.items()), 
    columns=["genre", "genre_id"]
)
genre_mapping_path = tracked_dir / "genre_mapping.csv"
genre_mapping_df.to_csv(genre_mapping_path, index=False)
print("Saved genre mapping to:", genre_mapping_path)
display(genre_mapping_df)

# Save the failed track IDs and errors for documentation
print("\nFailed conversions:", len(results.loc[results["status"] == "failed"]))
display(
    results.loc[
        results["status"] == "failed",
        ["genre", "split", "conversion_error"]
    ]
)

Training: 6393
Validation: 800
Test: 800
Saved, train_mel.csv: 6393 records -> ../data/tracked/train_mel.csv
Saved, val_mel.csv: 800 records -> ../data/tracked/val_mel.csv
Saved, test_mel.csv: 800 records -> ../data/tracked/test_mel.csv


genre,Electronic,Experimental,Folk,Hip-Hop,Instrumental,International,Pop,Rock
split,,,,,,,,
test,100,100,100,100,100,100,100,100
training,799,799,800,797,799,800,800,799
validation,100,100,100,100,100,100,100,100


Saved genre mapping to: ../data/tracked/genre_mapping.csv


,genre,genre_id
0,Electronic,0
1,Experimental,1
2,Folk,2
3,Hip-Hop,3
4,Instrumental,4
5,International,5
6,Pop,6
7,Rock,7



Failed conversions: 7


,genre,split,conversion_error
track_id,,,
98565,Hip-Hop,training,Unspecified internal error.
98567,Hip-Hop,training,Unspecified internal error.
98569,Hip-Hop,training,Unspecified internal error.
99134,Electronic,training,Error opening '../data/local/fma_small_mp3/fma...
107535,Instrumental,training,Audio is completely silent (all zeros).
108925,Rock,training,Error opening '../data/local/fma_small_mp3/fma...
133297,Experimental,training,Error opening '../data/local/fma_small_mp3/fma...


### Step 7. Create Standardized Image Loading Function for further processing

In [8]:
def load_mel_tensor(npy_path):
    """
    Loads a saved mel spectrogram, normalizes dB [-80, 0] to [0, 1] and adds a channel dimension: (128, 1292, 1).
    """
    mel = np.load(npy_path).astype(np.float32)
    # Normalize to [0, 1]
    normalized = (np.clip(mel, -80.0, 0.0) + 80.0) / 80.0
    return np.expand_dims(normalized, axis=-1)